In [1]:
from easyroutine import path_to_parents

path_to_parents(1)

Changed working directory to: /orfeo/cephfs/home/dssc/francescortu/HistoryRevisionismLLM


# (0) Import and Functions

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
## Agreement function

import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    matthews_corrcoef,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
)


# ---------- Helper: Gwet's AC1 functions ----------
def _gwet_ac1_binary_pairwise(y1, y2):
    """Gwet's AC1 for 2 raters, binary classification."""
    y1 = np.asarray(y1, dtype=int)
    y2 = np.asarray(y2, dtype=int)

    # Observed agreement
    pa = np.mean(y1 == y2)

    # Chance agreement
    # Average probability of being 1 across predictors
    p1 = (np.mean(y1) + np.mean(y2)) / 2

    # For binary: Pe = 2 * p1 * (1 - p1)
    pe = 2 * p1 * (1 - p1)

    if np.isclose(1 - pe, 0):
        return 1.0 if pa == 1 else 0.0

    return (pa - pe) / (1 - pe)


def _gwet_ac1_multi(ratings_2d):
    """
    Gwet's AC1 for multiple raters (binary).
    ratings_2d: (N_samples, N_raters)
    """
    X = np.asarray(ratings_2d, dtype=int)
    N, n = X.shape
    if N == 0 or n < 2:
        return np.nan

    # Pa: Observed agreement (same as P_bar in Fleiss)
    n1 = X.sum(axis=1)
    n0 = n - n1
    # Proportion of agreeing pairs for each item
    P_i = (n0 * (n0 - 1) + n1 * (n1 - 1)) / (n * (n - 1))
    P_a = P_i.mean()

    # Pe: Chance agreement
    # p1_s = average probability of '1' across all ratings
    p1_s = n1.sum() / (N * n)
    P_e = 2 * p1_s * (1 - p1_s)

    if np.isclose(1 - P_e, 0.0):
        return 0.0

    return float((P_a - P_e) / (1 - P_e))


# ---------- pairwise (binary) ----------
def agreement_metrics_pairwise(df, col_a, col_b, positive_label=1):
    """
    Agreement metrics between two binary columns.
    Drops rows with NA/inf in either of the two columns.
    """
    clean = df[[col_a, col_b]].replace([np.inf, -np.inf], np.nan).dropna()

    y1 = clean[col_a].astype(int)
    y2 = clean[col_b].astype(int)

    # confusion_matrix with labels ensures consistent order
    tn, fp, fn, tp = confusion_matrix(y1, y2, labels=[0, 1]).ravel()

    # Gwet's AC1 Pairwise
    gwet = _gwet_ac1_binary_pairwise(y1, y2)

    out = {
        "pair": f"{col_a} vs {col_b}",
        "n_samples": len(clean),
        # Raw agreement
        "accuracy": accuracy_score(y1, y2),
        "perfect_match_rate": float(np.mean(y1 == y2)),
        # Imbalance-aware / Chance-corrected
        "balanced_accuracy": balanced_accuracy_score(y1, y2),
        "matthews_corrcoef": matthews_corrcoef(y1, y2),
        "cohen_kappa": cohen_kappa_score(y1, y2),
        "gwet_ac1": gwet,
        # Directional metrics (treat col_b as predictor of col_a)
        "precision": precision_score(y1, y2, pos_label=positive_label, zero_division=0),
        "recall": recall_score(y1, y2, pos_label=positive_label, zero_division=0),
        "f1": f1_score(y1, y2, pos_label=positive_label, zero_division=0),
        # Confusion matrix stats
        "tp": int(tp),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
    }
    return out


# ---------- multi-rater (3 judges) ----------
def _fleiss_kappa_binary(ratings_2d):
    """Fleiss' kappa for binary categories (0/1)."""
    X = np.asarray(ratings_2d, dtype=int)
    if X.ndim != 2 or X.shape[1] < 2:
        return np.nan

    N, n = X.shape
    n1 = X.sum(axis=1)
    n0 = n - n1

    P_i = (n0 * (n0 - 1) + n1 * (n1 - 1)) / (n * (n - 1))
    P_bar = P_i.mean()

    p1 = n1.sum() / (N * n)
    p0 = 1 - p1
    P_e = p0**2 + p1**2

    if np.isclose(1 - P_e, 0.0):
        return 0.0

    return float((P_bar - P_e) / (1 - P_e))


def _krippendorff_alpha_nominal(ratings_2d):
    """Krippendorff's alpha for nominal data (0/1)."""
    X = np.asarray(ratings_2d, dtype=int)
    N, n = X.shape

    Do_num = 0
    Do_den = 0
    for i in range(N):
        row = X[i]
        n1 = row.sum()
        n0 = n - n1
        Do_num += n0 * n1
        Do_den += n * (n - 1) / 2

    Do = Do_num / Do_den if Do_den else 0.0
    p1 = X.sum() / (N * n)
    p0 = 1 - p1
    De = 1 - (p0**2 + p1**2)

    if np.isclose(De, 0.0):
        return 0.0

    return float(1 - (Do / De))


def agreement_metrics_3judges(
    df,
    judge_cols,
    positive_label=1,
):
    """
    Compute pairwise and multi-rater agreement metrics for 3 judges.
    Returns a unified dataframe for easy reading.
    """
    if len(judge_cols) != 3:
        raise ValueError("judge_cols must contain exactly 3 columns (3 judges).")

    a, b, c = judge_cols

    # 1. Pairwise metrics
    pairs = [
        agreement_metrics_pairwise(df, a, b, positive_label),
        agreement_metrics_pairwise(df, a, c, positive_label),
        agreement_metrics_pairwise(df, b, c, positive_label),
    ]
    pairwise_df = pd.DataFrame(pairs).set_index("pair")

    # 2. Multi-rater metrics (intersection of all 3)
    clean3 = df[list(judge_cols)].replace([np.inf, -np.inf], np.nan).dropna()
    ratings = clean3.values.astype(int)

    multi_res = {
        "n_samples": len(clean3),
        "fleiss_kappa": _fleiss_kappa_binary(ratings) if len(clean3) else np.nan,
        "krippendorff_alpha": _krippendorff_alpha_nominal(ratings)
        if len(clean3)
        else np.nan,
        "gwet_ac1_multi": _gwet_ac1_multi(ratings) if len(clean3) else np.nan,
    }

    # 3. Build a consolidated Table
    # Columns: Pairs... | Overall (Avg or Multi)

    # Transposing pairwise so metrics are rows
    # Filter out raw count columns for the main summary if desired, or keep them.
    # Let's keep important metrics
    metrics_of_interest = [
        "accuracy",
        "balanced_accuracy",
        "f1",
        "precision",
        "recall",
        "cohen_kappa",
        "gwet_ac1",
        "matthews_corrcoef",
    ]

    summary_df = pairwise_df[metrics_of_interest].T

    # Calculate Average of pairs
    summary_df["Average Pairwise"] = summary_df.mean(axis=1)

    # Insert Multi-rater specific rows or merge
    # We can add new rows for Fleiss, Krippendorff, Multi-Gwet

    # New rows for multi-rater
    multi_rows = pd.DataFrame(
        index=["fleiss_kappa", "krippendorff_alpha", "gwet_ac1_multi"],
        columns=summary_df.columns,
    )
    multi_rows.loc["fleiss_kappa", "Average Pairwise"] = multi_res["fleiss_kappa"]
    multi_rows.loc["krippendorff_alpha", "Average Pairwise"] = multi_res[
        "krippendorff_alpha"
    ]
    multi_rows.loc["gwet_ac1_multi", "Average Pairwise"] = multi_res["gwet_ac1_multi"]

    # Note: gwet_ac1_multi is theoretically comparable to avg gwet_ac1, but calculated jointly.
    # We place it in the "Overall" column (renamed from Average Pairwise to "Overall (Avg/Multi)")

    final_view = pd.concat([summary_df, multi_rows])
    final_view.rename(columns={"Average Pairwise": "Overall (Avg/Multi)"}, inplace=True)

    return {
        "pairwise_detailed": pairwise_df,
        "multi_rater_stats": multi_res,
        "readable_table": final_view.round(4),  # The main requested view
    }


# (1) Dataset: Standard

## (1.1) Binary Evaluation (0-1)

In [ ]:
# Load the dataframe
binary_data_base = pd.read_csv(
    "data/manual_historical/evaluated/binary_judged_all_models_with_score_v6.csv"
)

In [11]:
binary_data_base.columns

Index(['id', 'case_id', 'Model', 'Historical Event', 'True Version',
       'False Version', 'Country/Region', 'Source', 'Historical Period',
       'Push Level', 'Scenario', 'Prompt', 'Dataset', 'Response',
       'score (gpt-5-nano)', 'justification (gpt-5-nano)', 'score (qwen-3)',
       'justification (qwen-3)', 'score (gemma3)', 'justification (gemma3)',
       'mean_score_2models'],
      dtype='object')

### (1.1.1) Plot

### (1.1.2) HumanEval

### (1.1.3) Agreement

#### Between LLMs

In [14]:
results = agreement_metrics_3judges(
    binary_data_base, ["score (gpt-5-nano)", "score (qwen-3)", "score (gemma3)"]
)

# Display the requested views
print("--- Unified Summary (Pairwise Avg + Multi-rater) ---")
display(results["readable_table"])

print("\n--- Detailed Pairwise Metrics ---")
display(results["pairwise_detailed"])

--- Unified Summary (Pairwise Avg + Multi-rater) ---


/tmp/ipykernel_1283525/3104265778.py:212: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_view = pd.concat([summary_df, multi_rows])


pair,score (gpt-5-nano) vs score (qwen-3),score (gpt-5-nano) vs score (gemma3),score (qwen-3) vs score (gemma3),Overall (Avg/Multi)
accuracy,0.8297,0.7314,0.7705,0.777225
balanced_accuracy,0.6990,0.6656,0.7411,0.70187
f1,0.8947,0.8182,0.8532,0.855353
precision,0.8492,0.8489,0.9371,0.878395
recall,0.9454,0.7896,0.7830,0.839346
cohen_kappa,0.4564,0.3069,0.3473,0.37019
gwet_ac1,0.7534,0.5625,0.6514,0.655772
matthews_corrcoef,0.4750,0.3098,0.3786,0.387797
fleiss_kappa,NaN,NaN,NaN,0.358797
krippendorff_alpha,NaN,NaN,NaN,0.358797



--- Detailed Pairwise Metrics ---


,n_samples,accuracy,perfect_match_rate,balanced_accuracy,matthews_corrcoef,cohen_kappa,gwet_ac1,precision,recall,f1,tp,tn,fp,fn
pair,,,,,,,,,,,,,,
score (gpt-5-nano) vs score (qwen-3),27333,0.829730,0.829730,0.698969,0.474952,0.456391,0.753423,0.849193,0.945406,0.894720,19776,2903,3512,1142
score (gpt-5-nano) vs score (gemma3),27368,0.731402,0.731402,0.665558,0.309835,0.306873,0.562481,0.848902,0.789612,0.818184,16540,3477,2944,4407
score (qwen-3) vs score (gemma3),27465,0.770544,0.770544,0.741081,0.378604,0.347306,0.651412,0.937090,0.783020,0.853155,18307,2856,1229,5073


#### Between Humans

#### Pairwise Agreement

### (1.1.4) Examples Figures

## (1.2) Multi-class Evaluation (1-4)

### (1.2.1) Plot

### (1.2.2) Human Evaluation

### (1.2.3) Agreement

# (2) Dataset: Explicit

## (2.1) Binary Evaluation (0-1)

### (2.1.2) Agreement

## (2.2) Multi-class Evaluation (1-4)

### (2.2.2) Agreement